# 03 Feature Engineering for Women's Health Data

This notebook creates additional features from our daily cycle data to improve model performance.

## Features to Create:
- Rolling window statistics
- Lag features
- Cyclical encodings
- User-specific statistics
- Symptom encoding
- Temporal features

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_regression
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print("Libraries imported successfully!")

## 1. Load and Explore Data

In [ ]:
# Load the daily cycle data
df = pd.read_csv('data/processed/daily_data.csv')

print(f"Data shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nUnique users: {df['user_id'].str.split('_').str[:-1].str.join('_').nunique()}")
print(f"Total cycles: {df['user_id'].nunique()}")

# Display sample data
display(df.head(10))

In [ ]:
# Check data types and missing values
print("Data Info:")
print(df.info())

print("\nMissing values:")
missing_data = df.isnull().sum()
print(missing_data[missing_data > 0])

# Basic statistics
print("\nBasic statistics for target variables:")
display(df[['mood', 'energy', 'stress_level', 'cycle_day']].describe())

## 2. Cyclical Features

In [ ]:
def create_cyclical_features(df):
    """Create cyclical features for better cycle representation"""
    df_features = df.copy()
    
    # Cyclical encoding for cycle day (sine/cosine)
    df_features['cycle_day_sin'] = np.sin(2 * np.pi * df_features['cycle_day'] / df_features['cycle_length'])
    df_features['cycle_day_cos'] = np.cos(2 * np.pi * df_features['cycle_day'] / df_features['cycle_length'])
    
    # Cycle progress (0 to 1)
    df_features['cycle_progress'] = df_features['cycle_day'] / df_features['cycle_length']
    
    # Days from start/end
    df_features['days_from_start'] = df_features['cycle_day'] - 1
    df_features['days_to_end'] = df_features['cycle_length'] - df_features['cycle_day']
    
    # Quadratic cycle day (captures U-shaped patterns)
    normalized_day = (df_features['cycle_day'] - df_features['cycle_length']/2) / (df_features['cycle_length']/2)
    df_features['cycle_day_squared'] = normalized_day ** 2
    
    return df_features

df_with_cyclical = create_cyclical_features(df)
print(f"Added cyclical features. New shape: {df_with_cyclical.shape}")

In [ ]:
# Visualize cyclical features
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0,0].scatter(df_with_cyclical['cycle_day'], df_with_cyclical['cycle_day_sin'], alpha=0.1)
axes[0,0].set_title('Cycle Day Sin Encoding')
axes[0,0].set_xlabel('Cycle Day')

axes[0,1].scatter(df_with_cyclical['cycle_day'], df_with_cyclical['cycle_day_cos'], alpha=0.1)
axes[0,1].set_title('Cycle Day Cos Encoding')
axes[0,1].set_xlabel('Cycle Day')

axes[1,0].scatter(df_with_cyclical['cycle_progress'], df_with_cyclical['mood'], alpha=0.1)
axes[1,0].set_title('Mood vs Cycle Progress')
axes[1,0].set_xlabel('Cycle Progress')

axes[1,1].scatter(df_with_cyclical['cycle_day_squared'], df_with_cyclical['energy'], alpha=0.1)
axes[1,1].set_title('Energy vs Cycle Day Squared')
axes[1,1].set_xlabel('Cycle Day Squared')

plt.tight_layout()
plt.show()

## 3. Lag Features

In [ ]:
def create_lag_features(df, target_cols=['mood', 'energy', 'stress_level'], lags=[1, 2, 3, 7]):
    """Create lag features for target variables"""
    df_lag = df.copy()
    
    # Sort by user_id and cycle_day for proper lag calculation
    df_lag = df_lag.sort_values(['user_id', 'cycle_day'])
    
    for col in target_cols:
        for lag in lags:
            # Create lag features within each cycle (user_id)
            df_lag[f'{col}_lag_{lag}'] = df_lag.groupby('user_id')[col].shift(lag)
    
    return df_lag

df_with_lags = create_lag_features(df_with_cyclical)
print(f"Added lag features. New shape: {df_with_lags.shape}")

In [ ]:
# Check correlation between current mood and lagged mood
lag_cols = [col for col in df_with_lags.columns if 'lag' in col and 'mood' in col]
correlation_matrix = df_with_lags[['mood'] + lag_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title('Correlation: Current Mood vs Lagged Mood')
plt.tight_layout()
plt.show()

# Show lag feature statistics
print("\nLag feature availability:")
for col in lag_cols:
    non_null = df_with_lags[col].notna().sum()
    print(f"{col}: {non_null:,} values ({non_null/len(df_with_lags)*100:.1f}%)")

## 4. Rolling Window Statistics

In [ ]:
def create_rolling_features(df, target_cols=['mood', 'energy', 'stress_level'], windows=[3, 7, 14]):
    """Create rolling window statistics"""
    df_rolling = df.copy()
    
    # Sort by user_id and cycle_day
    df_rolling = df_rolling.sort_values(['user_id', 'cycle_day'])
    
    for col in target_cols:
        for window in windows:
            # Rolling statistics within each cycle
            grouped = df_rolling.groupby('user_id')[col]
            
            df_rolling[f'{col}_rolling_mean_{window}'] = grouped.rolling(window=window, min_periods=1).mean().reset_index(0, drop=True)
            df_rolling[f'{col}_rolling_std_{window}'] = grouped.rolling(window=window, min_periods=2).std().reset_index(0, drop=True)
            df_rolling[f'{col}_rolling_max_{window}'] = grouped.rolling(window=window, min_periods=1).max().reset_index(0, drop=True)
            df_rolling[f'{col}_rolling_min_{window}'] = grouped.rolling(window=window, min_periods=1).min().reset_index(0, drop=True)
    
    return df_rolling

df_with_rolling = create_rolling_features(df_with_lags)
print(f"Added rolling features. New shape: {df_with_rolling.shape}")

## 5. Symptom Encoding Features

In [ ]:
def create_symptom_features(df):
    """Create features from symptom data"""
    df_symptoms = df.copy()
    
    # Define all possible symptoms
    all_symptoms = ['cramps', 'headache', 'breast_tenderness', 'acne', 
                   'food_cravings', 'sleep_problems', 'not_defined']
    
    # Binary encoding for each symptom
    for symptom in all_symptoms:
        df_symptoms[f'has_{symptom}'] = df_symptoms['symptoms'].astype(str).str.contains(symptom, na=False).astype(int)
    
    # Count of symptoms per day
    df_symptoms['symptom_count'] = df_symptoms['symptoms'].astype(str).str.count(',') + 1
    df_symptoms['symptom_count'] = df_symptoms['symptom_count'].where(~df_symptoms['has_not_defined'], 0)
    
    # Symptom severity score (weighted)
    symptom_weights = {
        'cramps': 3, 'headache': 2, 'breast_tenderness': 2, 
        'acne': 1, 'food_cravings': 1, 'sleep_problems': 2, 'not_defined': 0
    }
    
    df_symptoms['symptom_severity'] = 0
    for symptom, weight in symptom_weights.items():
        df_symptoms['symptom_severity'] += df_symptoms[f'has_{symptom}'] * weight
    
    return df_symptoms

df_with_symptoms = create_symptom_features(df_with_rolling)
print(f"Added symptom features. New shape: {df_with_symptoms.shape}")

## 6. User-Specific Features

In [ ]:
def create_user_features(df):
    """Create user-specific statistical features"""
    df_user = df.copy()
    
    # Extract base user ID (without cycle number)
    df_user['base_user_id'] = df_user['user_id'].str.split('_').str[:-1].str.join('_')
    
    # User-level statistics across all their cycles
    user_stats = df_user.groupby('base_user_id').agg({
        'mood': ['mean', 'std'],
        'energy': ['mean', 'std'],
        'stress_level': ['mean', 'std'],
        'cycle_length': ['mean', 'std'],
        'symptom_severity': ['mean', 'max']
    }).round(3)
    
    # Flatten column names
    user_stats.columns = [f'user_{col[0]}_{col[1]}' for col in user_stats.columns]
    
    # Merge back to main dataframe
    df_user = df_user.merge(user_stats, left_on='base_user_id', right_index=True, how='left')
    
    # Deviation from user's normal patterns
    df_user['mood_deviation'] = df_user['mood'] - df_user['user_mood_mean']
    df_user['energy_deviation'] = df_user['energy'] - df_user['user_energy_mean']
    df_user['stress_deviation'] = df_user['stress_level'] - df_user['user_stress_level_mean']
    
    return df_user

df_with_user_features = create_user_features(df_with_symptoms)
print(f"Added user features. New shape: {df_with_user_features.shape}")

## 7. Final Feature Summary and Export

In [ ]:
# Final feature dataframe
df_final = df_with_user_features.copy()

print(f"Final dataset shape: {df_final.shape}")
print(f"\nFeature categories:")

# Count features by category
feature_categories = {
    'Original': len([col for col in df.columns if col in df_final.columns]),
    'Cyclical': len([col for col in df_final.columns if any(x in col for x in ['sin', 'cos', 'progress', 'squared'])]),
    'Lag': len([col for col in df_final.columns if 'lag' in col]),
    'Rolling': len([col for col in df_final.columns if 'rolling' in col]),
    'Symptom': len([col for col in df_final.columns if 'symptom' in col or 'has_' in col]),
    'User': len([col for col in df_final.columns if 'user_' in col or 'deviation' in col])
}

for category, count in feature_categories.items():
    print(f"  {category}: {count} features")

print(f"\nTotal features: {sum(feature_categories.values())}")

In [ ]:
# Correlation analysis of key features with targets
target_cols = ['mood', 'energy', 'stress_level']
important_features = ['cycle_day', 'cycle_progress', 'phase_encoded', 'symptom_count', 'symptom_severity']

# Add some lag and rolling features
lag_features = [col for col in df_final.columns if 'lag_1' in col][:6]
rolling_features = [col for col in df_final.columns if 'rolling_mean_3' in col][:6]

all_features = target_cols + important_features + lag_features + rolling_features
correlation_subset = df_final[all_features].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(correlation_subset, annot=False, cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix (Key Features)')
plt.tight_layout()
plt.show()

# Top correlations with mood
mood_corr = df_final.corr()['mood'].abs().sort_values(ascending=False)
print("\nTop 15 features correlated with mood:")
for i, (feature, corr) in enumerate(mood_corr.head(15).items()):
    print(f"{i+1:2d}. {feature:30s}: {corr:.3f}")

In [ ]:
# Clean and export the feature-engineered dataset
print("Cleaning and exporting dataset...")

# Handle any remaining infinite values
df_final = df_final.replace([np.inf, -np.inf], np.nan)

# Check final data quality
print(f"\nFinal data quality check:")
print(f"Shape: {df_final.shape}")
print(f"Missing values: {df_final.isnull().sum().sum():,}")
print(f"Infinite values: {np.isinf(df_final.select_dtypes(include=[np.number])).sum().sum()}")

# Save the feature-engineered dataset
output_path = 'data/processed/daily_data_with_features.csv'
df_final.to_csv(output_path, index=False)
print(f"\n✅ Feature-engineered dataset saved to: {output_path}")

# Save feature names for model training
feature_names = {
    'target_variables': target_cols,
    'categorical_features': ['cycle_phase', 'symptoms'] + [col for col in df_final.columns if 'has_' in col],
    'numerical_features': [col for col in df_final.columns if col not in target_cols + ['cycle_phase', 'symptoms', 'user_id', 'ClientID', 'original_index']],
    'user_features': [col for col in df_final.columns if 'user_' in col or 'deviation' in col],
    'temporal_features': [col for col in df_final.columns if any(x in col for x in ['lag', 'rolling', 'sin', 'cos', 'progress'])]
}

import json
with open('data/processed/feature_names.json', 'w') as f:
    # Convert sets to lists for JSON serialization
    feature_names_serializable = {k: list(v) if isinstance(v, (set, tuple)) else v for k, v in feature_names.items()}
    json.dump(feature_names_serializable, f, indent=2)

print(f"✅ Feature names saved to: data/processed/feature_names.json")
print(f"\n🎉 Feature engineering completed successfully!")

In [ ]:
# Final summary visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Feature count by category
categories = list(feature_categories.keys())
counts = list(feature_categories.values())
axes[0,0].bar(categories, counts, color='skyblue')
axes[0,0].set_title('Features by Category')
axes[0,0].set_ylabel('Number of Features')
axes[0,0].tick_params(axis='x', rotation=45)

# Missing data heatmap for key columns
key_columns = ['mood', 'energy', 'stress_level'] + [col for col in df_final.columns if 'lag_1' in col][:5]
missing_matrix = df_final[key_columns].isnull()
sns.heatmap(missing_matrix.head(1000), yticklabels=False, cbar=True, ax=axes[0,1])
axes[0,1].set_title('Missing Data Pattern (First 1000 rows)')

# Distribution of engineered features
axes[1,0].hist(df_final['cycle_progress'], bins=30, alpha=0.7, label='Cycle Progress')
axes[1,0].hist(df_final['symptom_severity'], bins=30, alpha=0.7, label='Symptom Severity')
axes[1,0].set_title('Distribution of Key Engineered Features')
axes[1,0].legend()

# Target variable relationships
sample_data = df_final.sample(5000)  # Sample for visualization
axes[1,1].scatter(sample_data['mood'], sample_data['energy'], alpha=0.3, label='Mood vs Energy')
axes[1,1].set_xlabel('Mood')
axes[1,1].set_ylabel('Energy')
axes[1,1].set_title('Target Variable Relationships')

plt.tight_layout()
plt.show()

print("\n📊 Feature engineering visualization complete!")
print(f"Dataset ready for model training with {df_final.shape[1]} features and {df_final.shape[0]:,} samples.")